In [2]:
pip install langchain_huggingface


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install langchain_community

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import spacy
import contractions
from textblob import TextBlob

c:\Users\sushm\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


### load the document

In [5]:
data=open("data.txt").read()

In [6]:
data = data.lower()

### Removing extra space

In [7]:
import re
data=re.sub(r'\s{2,}','',data )

removing numbers like 1 ,2,3

In [8]:
#data=re.sub(r'')

### contractions

In [9]:
contractions.fix(data)


'machine learning is the science of teaching computers to learn from data.\nit is a central branch of artificial intelligence.\nunlike traditional programming, ml does not rely on explicit rules.\ninstead, it infers rules from examples.\nthe essence of ml lies in generalization.\na model trained on past data must perform well on unseen data.\nthis ability makes ml powerful and versatile.\nml begins with data collection.\ndata can be structured, semi-structured, or unstructured.\nstructured data includes tables in sql.\nsemi-structured data includes logs or json files.\nunstructured data includes images, audio, and text.\nthe richness of data determines model capability.\nalgorithms process this data to uncover relationships.\nlinear regression models simple numeric relationships.\ndecision trees split data into hierarchical rules.\nsupport vector machines find optimal boundaries.\nneural networks mimic the human brain.\ndeep learning uses multiple layers of neural networks.\nit powers 

4.Removing punctuation and special characters

In [10]:
data=re.sub(r'[^0-9a-zA-Z\s]','',data)

### Textblob

In [11]:
# values=TextBlob(data).correct()
# values

### spacy

### lemmatization

In [12]:
import spacy
nlp = spacy.load('en_core_web_sm')
tokens=nlp(data)
updated_tokens=[token.lemma_ for token  in tokens if not token.is_stop]
data = ' '.join(updated_tokens).strip()
data

'machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text \n richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule \n support vector machine find optimal boundary \n neural network mimic human brain \n deep learning use multiple layer neural network \n power breakthrough vision language \n ml divide supervise learning \n divide unsupervised learning \n reinforcement learning form paradigm \n supervise learning use

### chunking(converting docs into chunks)

In [13]:
splitter=text_splitter=RecursiveCharacterTextSplitter(
                  chunk_size=200,
                  chunk_overlap=40,
)

chunks=splitter.create_documents([data])
chunks
#print(chunks[0])
#type(chunks)
print(chunks[0].page_content)


machine learning science teach computer learn datum 
 central branch artificial intelligence 
 unlike traditional programming ml rely explicit rule 
 instead infer rule example


In [14]:
print(chunks)

[Document(metadata={}, page_content='machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example'), Document(metadata={}, page_content='instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection'), Document(metadata={}, page_content='ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text'), Document(metadata={}, page_content='richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule'), Document(metadata={}, page_content='support vector machine find optimal boundary \n neural networ

In [15]:
chunks[0].metadata={'file_name':{'data.txt'}}
chunks

[Document(metadata={'file_name': {'data.txt'}}, page_content='machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example'),
 Document(metadata={}, page_content='instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection'),
 Document(metadata={}, page_content='ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text'),
 Document(metadata={}, page_content='richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule'),
 Document(metadata={}, page_content='support vector machine find opti

### embeddings(coverts chunks into vectors)

In [16]:
print(chunks)

[Document(metadata={'file_name': {'data.txt'}}, page_content='machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example'), Document(metadata={}, page_content='instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection'), Document(metadata={}, page_content='ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text'), Document(metadata={}, page_content='richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule'), Document(metadata={}, page_content='support vector machine find optimal 

In [17]:
chunks

[Document(metadata={'file_name': {'data.txt'}}, page_content='machine learning science teach computer learn datum \n central branch artificial intelligence \n unlike traditional programming ml rely explicit rule \n instead infer rule example'),
 Document(metadata={}, page_content='instead infer rule example \n essence ml lie generalization \n model train past datum perform unseen datum \n ability make ml powerful versatile \n ml begin data collection'),
 Document(metadata={}, page_content='ml begin data collection \n datum structure semistructure unstructured \n structured datum include table sql \n semistructure datum include log json file \n unstructured datum include image audio text'),
 Document(metadata={}, page_content='richness datum determine model capability \n algorithm process datum uncover relationship \n linear regression model simple numeric relationship \n decision tree split datum hierarchical rule'),
 Document(metadata={}, page_content='support vector machine find opti

In [18]:
embeddings_model=HuggingFaceEmbeddings(
     model_name='sentence-transformers/all-miniLM-L6-V2'
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [19]:
vectordb=FAISS.from_documents(
    documents=chunks,
    embedding=embeddings_model
    
)
vectordb

In [20]:
user_query='what is machine learning ?'
r_chunks=vectordb.similarity_search(user_query)

In [21]:
updated_r_chunks=set()
for chunk in r_chunks:
    updated_r_chunks.add(chunk.page_content)
updated_r_chunks
R_text='\n'.join(updated_r_chunks)
R_text

'machine learning create \n machine learning innovate \n machine learning discover \n machine learning imagine \n machine learning evolve \n machine learning progress \n machine learning succeed\nmachine learning discover \n machine learning imagine \n machine learning evolve \n machine learning progress \n machine learning succeed \n machine learning thrive \n machine learning flourish\nmachine learning serve \n machine learning protect \n machine learning secure \n machine learning save \n machine learning healing \n machine learning guide \n machine learning lead\nmachine learning unite \n machine learning guide \n machine learning shape \n machine learning define \n machine learning create \n machine learning innovate \n machine learning discover'

In [22]:
def r_search(query,k=2):
        R_chunks=vectordb.similarity_search(query,k=k)
        R_chunks={doc.page_content for doc in R_chunks}
        R_text='\n'.join(R_chunks)
        #print(R_chunks)
        return R_chunks
    
def g_text(r_search,query):
        import os
        prompt =f'''
              you are an helpfull assistant
              Assigned task for you: Structure my output => {r_search}
              for this input => {query}
            note:
            1)don't add extra contents just structure mentioned output.
            2)if there is any mistakes in output correct or else keep the original output with structured result
            with structured result.
            Output structure:
            Input:{query}
            Output:structure output'''
        
        llm_model=ChatGoogleGenerativeAI(
               model="gemini-2.5-flash"   #gemini-3.5-flash
          )
        response=llm_model.invoke(prompt).content 
        return response
    

user_prompt='Explain Machine Learning'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
r_response =r_search(user_prompt)
g_response = g_text(r_response,user_prompt)
print(g_response)




# response = rag_query(user_prompt)
# print(response)

Input:Explain Machine Learning
Output:{'machine learning create \n machine learning innovate \n machine learning discover \n machine learning imagine \n machine learning evolve \n machine learning progress \n machine learning succeed', 'machine learning unite \n machine learning guide \n machine learning shape \n machine learning define \n machine learning create \n machine learning innovate \n machine learning discover'}
